## Step 1 - Install and Import Libraries

In [ ]:
# Importing all the libraries we need
import pandas as pd
import numpy as np
import ast
import warnings
warnings.filterwarnings('ignore')

# For graphs and charts
import matplotlib.pyplot as plt
import seaborn as sns

# For building the ML model
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

print("Done! All libraries imported.")

## Step 2 - Load the Dataset

In [ ]:
# We need to upload the CSV file from our computer
from google.colab import files
uploaded = files.upload()  # A button will appear - click it and upload tmdb_5000_movies.csv

# Reading the CSV into a dataframe
df = pd.read_csv('tmdb_5000_movies.csv')

print("Dataset loaded!")
print("Number of movies:", len(df))
print("Columns:", list(df.columns))

In [ ]:
# Let's look at the first few rows
df.head()

## Step 3 - Clean the Data

The dataset has some issues:
- Some movies have budget or revenue as 0 (missing data)
- The genres column is in a special JSON format, we need to extract it

We fix all of this below.

In [ ]:
# Remove movies where budget or revenue is 0 because those are incomplete entries
df = df[(df['budget'] > 0) & (df['revenue'] > 0)]
print("Movies remaining after removing 0 budget/revenue:", len(df))

In [ ]:
# The genres column looks like this: [{"id": 28, "name": "Action"}, ...]
# We need to extract just the genre names

def get_genres(text):
    try:
        genre_list = ast.literal_eval(text)
        names = [g['name'] for g in genre_list]
        return ', '.join(names)  # join them like "Action, Adventure"
    except:
        return ''

df['genre_names'] = df['genres'].apply(get_genres)

# Let's see what it looks like now
df[['title', 'genre_names']].head(5)

In [ ]:
# Create the target column - Hit (1) or Flop (0)
df['hit'] = (df['revenue'] >= 2 * df['budget']).astype(int)

# Check how many hits and flops we have
print("Hit (1) = Revenue >= 2x Budget")
print("Flop (0) = Revenue < 2x Budget")
print()
print(df['hit'].value_counts())

## Step 4 - Feature Engineering

We can't use the genre names directly in a model. We need to convert them into numbers.

We will create one column per genre, with 1 if the movie has that genre and 0 if it doesn't.
This is called **one-hot encoding** (or multi-hot encoding here since a movie can have multiple genres).

In [ ]:
# List of genres we will use
genres_to_use = ['Action', 'Comedy', 'Drama', 'Horror',
                 'Romance', 'Thriller', 'Animation', 'Adventure']

# Create a 0/1 column for each genre
for genre in genres_to_use:
    df['is_' + genre] = df['genre_names'].apply(
        lambda x: 1 if genre in x else 0
    )

# Also extract the release year from release_date
df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')
df['release_year'] = df['release_date'].dt.year.fillna(2000).astype(int)

print("New columns added!")
df[['title', 'is_Action', 'is_Comedy', 'is_Drama', 'release_year']].head(5)

In [ ]:
# Select the final features we will use to train our model
feature_columns = ['budget', 'popularity', 'runtime', 'vote_average',
                   'release_year',
                   'is_Action', 'is_Comedy', 'is_Drama', 'is_Horror',
                   'is_Romance', 'is_Thriller', 'is_Animation', 'is_Adventure']

# Drop rows with any missing values in our selected columns
df_clean = df[feature_columns + ['hit', 'title']].dropna()

print("Final dataset size:", len(df_clean))
print("Features used:", feature_columns)

## Step 5 - Exploratory Data Analysis (EDA)

Before building the model, let's explore the data visually.

In [ ]:
# Chart 1: How many hits vs flops?
plt.figure(figsize=(6, 4))
counts = df_clean['hit'].value_counts()
plt.bar(['Flop', 'Hit'], [counts[0], counts[1]], color=['salmon', 'mediumseagreen'], width=0.4)
plt.title('Number of Hits vs Flops')
plt.ylabel('Count')
plt.text(0, counts[0] + 5, str(counts[0]), ha='center', fontweight='bold')
plt.text(1, counts[1] + 5, str(counts[1]), ha='center', fontweight='bold')
plt.show()

In [ ]:
# Chart 2: Does higher budget mean more hits?
plt.figure(figsize=(8, 4))
plt.hist(df_clean[df_clean['hit'] == 1]['budget'] / 1e6, bins=30,
         alpha=0.6, color='mediumseagreen', label='Hit')
plt.hist(df_clean[df_clean['hit'] == 0]['budget'] / 1e6, bins=30,
         alpha=0.6, color='salmon', label='Flop')
plt.xlabel('Budget (in Millions $)')
plt.ylabel('Number of Movies')
plt.title('Budget Distribution - Hit vs Flop')
plt.legend()
plt.show()

In [ ]:
# Chart 3: Average rating of hits vs flops
avg_rating = df_clean.groupby('hit')['vote_average'].mean()
plt.figure(figsize=(6, 4))
plt.bar(['Flop', 'Hit'], [avg_rating[0], avg_rating[1]],
        color=['salmon', 'mediumseagreen'], width=0.4)
plt.title('Average Audience Rating - Hit vs Flop')
plt.ylabel('Average Vote')
plt.ylim(0, 10)
for i, v in enumerate([avg_rating[0], avg_rating[1]]):
    plt.text(i, v + 0.1, f'{v:.2f}', ha='center', fontweight='bold')
plt.show()

## Step 6 - Train the Model

We will try two models:
1. **Logistic Regression** - a simple and commonly used classification algorithm
2. **Random Forest** - a more powerful model that uses multiple decision trees

In [ ]:
# Separate features (X) and target (y)
X = df_clean[feature_columns]
y = df_clean['hit']

# Split into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training samples:", len(X_train))
print("Testing samples :", len(X_test))

In [ ]:
# Scale the features - brings all values to a similar range
# This is important especially for Logistic Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

In [ ]:
# Train Model 1: Logistic Regression
lr_model = LogisticRegression(max_iter=500)
lr_model.fit(X_train_scaled, y_train)

lr_predictions = lr_model.predict(X_test_scaled)
lr_accuracy = accuracy_score(y_test, lr_predictions)
print("Logistic Regression Accuracy:", round(lr_accuracy * 100, 2), "%")

In [ ]:
# Train Model 2: Random Forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_scaled, y_train)

rf_predictions = rf_model.predict(X_test_scaled)
rf_accuracy = accuracy_score(y_test, rf_predictions)
print("Random Forest Accuracy:", round(rf_accuracy * 100, 2), "%")

## Step 7 - Evaluate the Models

In [ ]:
# Confusion Matrix for both models
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, model_name, preds in zip(axes,
                                  ['Logistic Regression', 'Random Forest'],
                                  [lr_predictions, rf_predictions]):
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Flop', 'Hit'],
                yticklabels=['Flop', 'Hit'])
    ax.set_title(model_name)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.suptitle('Confusion Matrices', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nThe confusion matrix shows:")
print("- Top-left: Correctly predicted Flops")
print("- Bottom-right: Correctly predicted Hits")
print("- Top-right and Bottom-left: Wrong predictions")

In [ ]:
# Classification Report for Random Forest (our better model)
print("Detailed Report - Random Forest:")
print(classification_report(y_test, rf_predictions, target_names=['Flop', 'Hit']))

In [ ]:
# Which features are most important according to Random Forest?
feature_importance = pd.Series(rf_model.feature_importances_, index=feature_columns)
feature_importance = feature_importance.sort_values(ascending=True)

plt.figure(figsize=(8, 6))
feature_importance.plot(kind='barh', color='steelblue')
plt.title('Feature Importance - Which factors matter the most?')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

## Step 8 - Predict a New Movie

Now that the model is trained, we can use it to predict any movie we want.
Just change the values below and run the cell.

In [ ]:
# ---- CHANGE THESE VALUES ----
movie_budget     = 50000000   # budget in dollars (50 million = 50000000)
movie_popularity = 80         # popularity score (check TMDB for reference)
movie_runtime    = 120        # runtime in minutes
movie_rating     = 7.5        # expected audience rating out of 10
movie_year       = 2024       # release year

# Genre - set 1 if the movie belongs to that genre, 0 if not
is_Action    = 1
is_Comedy    = 0
is_Drama     = 0
is_Horror    = 0
is_Romance   = 0
is_Thriller  = 1
is_Animation = 0
is_Adventure = 1
# ------------------------------

# Put all values together in the correct order
new_movie = [[
    movie_budget,
    movie_popularity,
    movie_runtime,
    movie_rating,
    movie_year,
    is_Action, is_Comedy, is_Drama, is_Horror,
    is_Romance, is_Thriller, is_Animation, is_Adventure
]]

# Scale the input using the same scaler we trained on
new_movie_scaled = scaler.transform(new_movie)

# Get the prediction
prediction = rf_model.predict(new_movie_scaled)[0]
probability = rf_model.predict_proba(new_movie_scaled)[0]

hit_chance  = round(probability[1] * 100, 1)
flop_chance = round(probability[0] * 100, 1)

# Show the result
print("==============================")
if prediction == 1:
    print("Prediction: HIT")
else:
    print("Prediction: FLOP")
print(f"Hit  chance : {hit_chance}%")
print(f"Flop chance : {flop_chance}%")
print("==============================")